# Buzzing Intelligence: Acoustic Classification of Agricultural Insects

#### Downloading the Dataset through kagglehub

In [ ]:
import kagglehub

# Download latest version of dataset
path = kagglehub.dataset_download("hesi0ne/insectsound1000")
print("Path to dataset files:", path)

#### Generating the Log-Mel Spectrograms

In [ ]:
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
from tqdm import tqdm

# Set up directory paths
dataset_dir = Path("InsectSound1000")
output_dir = Path("LogMelSpectrograms")
output_dir.mkdir(parents=True, exist_ok=True)

# Set up audio and log-mel spectrogram parameters
SAMPLE_RATE = 22050
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
FMIN = 80
FMAX = 12000

# Find the WAV files
wav_files = sorted(dataset_dir.rglob("*.wav"))
print(f"Found {len(wav_files)} WAV files")

# Process the files
for wav_path in tqdm(wav_files):

    try:
        audio, sr = sf.read(wav_path)

        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        audio = audio.astype(np.float32)

        if sr != SAMPLE_RATE:
            audio = librosa.resample(
                audio,
                orig_sr=sr,
                target_sr=SAMPLE_RATE
            )
            sr = SAMPLE_RATE

        # Perform peak normalization and preemphasis
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak
        audio = librosa.effects.preemphasis(audio)

        # Compute for Mel spectrograms
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=sr,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            win_length=WIN_LENGTH,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=FMAX,
            power=2.0
        )

        # Convert to log-Mel spectrograms
        log_mel = librosa.power_to_db(
            mel,
            ref=1.0,
            top_db=80
        )
        log_mel = log_mel.astype(np.float32)

        # Save outputs to directory
        relative_path = wav_path.relative_to(dataset_dir)
        save_path = (
            output_dir / relative_path
        ).with_suffix(".npy")
        save_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        np.save(save_path, log_mel)

    except Exception as e:
        print(f"Error processing {wav_path}: {e}")

print("Done.")

#### Generating TFRecords

In [ ]:
from pathlib import Path
import random
import math
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

import tensorflow as tf

# Set up directories and parameters
DATA_DIR = Path("LogMelSpectrograms")
OUTPUT_DIR = Path("TFRecords")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)
SHARD_SIZE = 1000
SEED = 42

# Set up random seed for reproducibility
random.seed(SEED)
np.random.seed(SEED)

# Collect files
files = sorted(DATA_DIR.rglob("*.npy"))
print(f"Found {len(files)} files")

# Debug sample
sample = np.load(files[0])
print("\nSample Info")
print("Shape:", sample.shape)
print("Min:", sample.min())
print("Max:", sample.max())
print("Mean:", sample.mean())

# Generate metadata based on filename
records = []
for path in files:
    stem = path.stem
    parts = stem.split("_")

    # Set up group_id for group-aware splits later on
    group_id = "_".join(parts[:4])

    species = "_".join(parts[1:3])
    records.append({
        "path": str(path),
        "species": species,
        "group": group_id
    })

df = pd.DataFrame(records)

# Get labels
species_to_idx = {
    s: i for i, s in enumerate(
        sorted(df["species"].unique())
    )
}
idx_to_species = {
    i: s for s, i in species_to_idx.items()
}
df["label"] = df["species"].map(
    species_to_idx
)
NUM_CLASSES = len(species_to_idx)
print(f"\nClasses: {NUM_CLASSES}")

# Perform stratified group-aware train-validation-test split
# (approximately 70%-15%-15%)
sgkf_test = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=SEED
)
train_val_idx, test_idx = next(
    sgkf_test.split(
        df,
        y=df["label"],
        groups=df["group"]
    )
)
train_val_df = df.iloc[train_val_idx]
test_df = df.iloc[test_idx]

sgkf_val = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=SEED
)
train_idx, val_idx = next(
    sgkf_val.split(
        train_val_df,
        y=train_val_df["label"],
        groups=train_val_df["group"]
    )
)
train_df = train_val_df.iloc[train_idx]
val_df = train_val_df.iloc[val_idx]

# Print summary of the splits
print("\n=================================================")
print("SPLIT SIZES")
print("=================================================")
print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")
print("\nPercentages:")
total = len(df)
print(f"Train: {100 * len(train_df)/total:.2f}%")
print(f"Val:   {100 * len(val_df)/total:.2f}%")
print(f"Test:  {100 * len(test_df)/total:.2f}%")

# Verify species distribution
print("\n=================================================")
print("SPECIES DISTRIBUTION")
print("=================================================")

for split_name, split_df in [
    ("TRAIN", train_df),
    ("VAL", val_df),
    ("TEST", test_df)
]:
    print(f"\n{split_name}")
    counts = (
        split_df["species"]
        .value_counts()
        .sort_index()
    )
    print(counts)

# Verify group leakage
train_groups = set(train_df["group"])
val_groups = set(val_df["group"])
test_groups = set(test_df["group"])

print("\n=================================================")
print("GROUP LEAKAGE CHECK")
print("=================================================")
print(
    "Train-Val overlap:",
    len(train_groups & val_groups)
)
print(
    "Train-Test overlap:",
    len(train_groups & test_groups)
)
print(
    "Val-Test overlap:",
    len(val_groups & test_groups)
)

# Define helper functions
def bytes_feature(value):

    return tf.train.Feature(
        bytes_list=tf.train.BytesList(value=[value])
    )
def int64_feature(value):

    return tf.train.Feature(
        int64_list=tf.train.Int64List(value=[value])
    )
def serialize_example(path, label):
    x = np.load(path)
    x = np.nan_to_num(
        x,
        nan=-80.0,
        posinf=0.0,
        neginf=-80.0
    )
    x = np.clip(
        x,
        -80.0,
        0.0
    )
    if x.ndim != 2:

        raise ValueError(
            f"Expected 2D log-mel, got {x.shape}"
        )
    x = np.expand_dims(
        x,
        axis=-1
    )
    x = x.astype(np.float32)
    feature = {
        "spectrogram": bytes_feature(
            x.tobytes()
        ),
        "label": int64_feature(
            int(label)
        ),
        "height": int64_feature(
            x.shape[0]
        ),
        "width": int64_feature(
            x.shape[1]
        ),
        "channels": int64_feature(
            x.shape[2]
        )
    }
    example = tf.train.Example(
        features=tf.train.Features(
            feature=feature
        )
    )
    return example.SerializeToString()

# Write split
def write_split(dataframe, split_name):
    split_dir = OUTPUT_DIR / split_name
    split_dir.mkdir(
        parents=True,
        exist_ok=True
    )
    total = len(dataframe)
    num_shards = math.ceil(
        total / SHARD_SIZE
    )
    print(f"\nWriting {split_name}")
    print(f"Samples: {total}")
    print(f"Shards:  {num_shards}")

    # Shuffle rows
    dataframe = dataframe.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    # Write shards
    for shard_idx in range(num_shards):
        start = shard_idx * SHARD_SIZE
        end = start + SHARD_SIZE
        shard_df = dataframe.iloc[start:end]
        shard_path = split_dir / (
            f"{split_name}-{shard_idx:05d}.tfrec"
        )
        with tf.io.TFRecordWriter(
            str(shard_path)
        ) as writer:
            for row in shard_df.itertuples():
                example = serialize_example(
                    row.path,
                    row.label
                )
                writer.write(example)
        print(
            f"Wrote {shard_path.name} "
            f"({len(shard_df)} samples)"
        )

# Create TFRecords
write_split(train_df, "train")
write_split(val_df, "val")
write_split(test_df, "test")

print("\n=================================================")
print("DONE")
print("=================================================")

#### Prepare Datasets for Training, Validation, and Testing

In [ ]:
from pathlib import Path
import random
import numpy as np
import keras
import tensorflow as tf

from sklearn.metrics import classification_report

# Set up directories and parameters
TFRECORD_DIR = Path("/content/TFRecords")
SEED = 42
BATCH_SIZE = 32
EPOCHS = 30
HEIGHT = 128
WIDTH = 256
NUM_CLASSES = 12
AUTOTUNE = tf.data.AUTOTUNE

# Set up random seed for reproducibility
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

# Get TFRecord files
train_files = sorted((TFRECORD_DIR / "train").glob("*.tfrec"))
val_files = sorted((TFRECORD_DIR / "val").glob("*.tfrec"))
test_files = sorted((TFRECORD_DIR / "test").glob("*.tfrec"))

print(f"Train shards: {len(train_files)}")
print(f"Val shards:   {len(val_files)}")
print(f"Test shards:  {len(test_files)}")

feature_description = {
    "spectrogram": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
    "height": tf.io.FixedLenFeature([], tf.int64),
    "width": tf.io.FixedLenFeature([], tf.int64),
    "channels": tf.io.FixedLenFeature([], tf.int64)
}

# Parse TFRecords
def parse_example(example_proto):
    example = tf.io.parse_single_example(
        example_proto,
        feature_description
    )
    x = tf.io.decode_raw(
        example["spectrogram"],
        tf.float32
    )
    height = tf.cast(example["height"], tf.int32)
    width = tf.cast(example["width"], tf.int32)
    channels = tf.cast(example["channels"], tf.int32)
    x = tf.reshape(
        x,
        (height, width, channels)
    )
    label = tf.cast(
        example["label"],
        tf.int32
    )
    return x, label

# Define function to fix the shapes
def random_crop_or_pad(x):
    width = tf.shape(x)[1]
    def pad():
        pad_amount = WIDTH - width
        return tf.pad(
            x,
            [[0, 0], [0, pad_amount], [0, 0]],
            constant_values=-80.0
        )
    def crop():
        start = tf.random.uniform(
            [],
            minval=0,
            maxval=width - WIDTH + 1,
            dtype=tf.int32
        )
        return x[:, start:start + WIDTH, :]
    return tf.cond(
        width < WIDTH,
        pad,
        crop
    )

def center_crop_or_pad(x):
    width = tf.shape(x)[1]
    def pad():
        pad_amount = WIDTH - width
        return tf.pad(
            x,
            [[0, 0], [0, pad_amount], [0, 0]],
            constant_values=-80.0
        )
    def crop():
        start = (width - WIDTH) // 2
        return x[:, start:start + WIDTH, :]
    return tf.cond(
        width < WIDTH,
        pad,
        crop
    )

# Define data augmentation function for use in training
def spec_augment(x):
    h = tf.shape(x)[0]
    w = tf.shape(x)[1]
    
    # Time mask   
    if tf.random.uniform([]) < 0.5:
        t = tf.random.uniform(
            [],
            minval=10,
            maxval=30,
            dtype=tf.int32
        )
        t = tf.minimum(t, w - 1)
        t0 = tf.random.uniform(
            [],
            minval=0,
            maxval=w - t,
            dtype=tf.int32
        )
        mask = tf.concat(
            [
                tf.ones((h, t0, 1)),
                tf.zeros((h, t, 1)),
                tf.ones((h, w - t0 - t, 1))
            ],
            axis=1
        )
        x *= mask

    # Frequency mask
    if tf.random.uniform([]) < 0.5:
        f = tf.random.uniform(
            [],
            minval=5,
            maxval=15,
            dtype=tf.int32
        )
        f = tf.minimum(f, h - 1)
        f0 = tf.random.uniform(
            [],
            minval=0,
            maxval=h - f,
            dtype=tf.int32
        )
        mask = tf.concat(
            [
                tf.ones((f0, w, 1)),
                tf.zeros((f, w, 1)),
                tf.ones((h - f0 - f, w, 1))
            ],
            axis=0
        )
        x *= mask

    return x

# Data preprocessing function
def preprocess(x, y, training=False):
    x = tf.cast(x, tf.float32)
    if training:
        x = random_crop_or_pad(x)
    else:
        x = center_crop_or_pad(x)

    mean = tf.reduce_mean(x)
    std = tf.math.reduce_std(x)
    x = (x - mean) / (std + 1e-6)

    if training:
        x = spec_augment(x)
    return x, y

# Create datasets
def make_dataset(files, training=False):
    ds = tf.data.TFRecordDataset(
        [str(f) for f in files],
        num_parallel_reads=AUTOTUNE
    )
    ds = ds.map(
        parse_example,
        num_parallel_calls=AUTOTUNE
    )
    if training:
        ds = ds.shuffle(
            10000,
            reshuffle_each_iteration=True
        )
    ds = ds.map(
        lambda x, y: preprocess(x, y, training),
        num_parallel_calls=AUTOTUNE
    )
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds

train_ds = make_dataset(train_files, training=True)
val_ds = make_dataset(val_files, training=False)
test_ds = make_dataset(test_files, training=False)

#### Build and Train CNN Model

In [ ]:
inputs = keras.Input(shape=(HEIGHT, WIDTH, 1))

# Block 1
x = keras.layers.Conv2D(32, 8, padding="same")(inputs)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.MaxPooling2D()(x)

# Block 2
x = keras.layers.Conv2D(64, 8, padding="same")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.MaxPooling2D()(x)

# Block 3
x = keras.layers.Conv2D(128, 8, padding="same")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.MaxPooling2D()(x)

# Block 4
x = keras.layers.Conv2D(128, 8, padding="same")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.GlobalAveragePooling2D()(x)

# Classifier
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = keras.Model(inputs, outputs)

# Compile model
model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        )
    ]
)

model.summary()

# Set up callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# Train CNN model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# Evaluate CNN model
test_loss, test_acc, test_top3 = model.evaluate(
    test_ds,
    verbose=1
)

print("\n=================================================")
print("FINAL TEST RESULTS")
print("=================================================")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Top3: {test_top3:.4f}")

# Make predictions
y_true = []
y_pred = []

for X_batch, y_batch in test_ds:

    preds = model.predict(X_batch, verbose=0)
    preds = np.argmax(preds, axis=1)

    y_true.extend(y_batch.numpy().tolist())
    y_pred.extend(preds.tolist())

# Print classification report
print("\nClassification Report:\n")
print(
    classification_report(
        y_true,
        y_pred,
        digits=4
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()